# NB1 -- Data Generator
## Computes all metrics and saves to `Tables_datas/`

**D. Sierra-Porta & Varotsos** (2025)

---
Run this notebook ONCE (or when data/parameters change).
NB2 loads everything from `Tables_datas/` -- no recomputation needed.

| Output file | Contents |
|-------------|----------|
| `Tables_datas/static_mfdfa.csv` | h(2), Delta-alpha, q_stable per station |
| `Tables_datas/rolling_mfdfa_{ST}.csv` | h(2), Delta-alpha daily rolling (all stations) |
| `Tables_datas/rolling_nta_ATHN.csv` | kappa1, S_nta daily rolling (ATHN, on raw counts) |
| `Tables_datas/rolling_nta_JUNG.csv` | kappa1, S_nta daily rolling (JUNG, on raw counts) |
| `Tables_datas/combined_ATHN.csv` | Merged rolling MF-DFA + NTA (ATHN) |
| `Tables_datas/combined_JUNG.csv` | Merged rolling MF-DFA + NTA (JUNG) |
| `Tables_datas/fd_parameters.csv` | All metrics at each FD onset (9 events) |


## 0. Setup and imports

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

import fathon
from fathon import fathonUtils as fu

os.makedirs('Tables_datas', exist_ok=True)
print(f'fathon {fathon.__version__} ready.')
print('Output folder: Tables_datas/')

# ── Forbush event dates (main targets) ────────────────────────────────────────
FD1 = pd.Timestamp('2024-03-24')
FD2 = pd.Timestamp('2024-05-10')

# ── FD catalogue: NMDB, magnitude > 6%, within 2010-2025 ─────────────────────
FD_EVENTS = [
    {'date': pd.Timestamp('2011-10-24'), 'magnitude': 6.01},
    {'date': pd.Timestamp('2012-03-08'), 'magnitude': 10.40},
    {'date': pd.Timestamp('2012-07-14'), 'magnitude': 6.70},
    {'date': pd.Timestamp('2015-06-22'), 'magnitude': 8.57},
    {'date': pd.Timestamp('2017-09-07'), 'magnitude': 6.41},
    {'date': pd.Timestamp('2021-11-03'), 'magnitude': 9.68},
    {'date': pd.Timestamp('2023-04-23'), 'magnitude': 6.57},
    {'date': pd.Timestamp('2024-03-24'), 'magnitude': 11.47},
    {'date': pd.Timestamp('2024-05-10'), 'magnitude': 15.32},
]
df_fd = pd.DataFrame(FD_EVENTS).set_index('date')
print(f'FD catalogue loaded: {len(df_fd)} events')


fathon 1.3.3 ready.
Output folder: Tables_datas/
FD catalogue loaded: 9 events


## 1. Data loading and preprocessing

In [2]:
df = pd.read_csv(
    'daily_mean_NMDBallstations_missing_lessthan5_2010-2025.csv',
    parse_dates=True
)
df['start_date_time'] = pd.to_datetime(df['start_date_time'])
df.set_index('start_date_time', inplace=True)
df = df.drop(columns=['MOSC'])  # instrumental level-shift ~2016

df_clean = df.interpolate(method='linear', limit=10).ffill().bfill()
STATIONS = list(df_clean.columns)
N = len(df_clean)

print(f'Date range : {df_clean.index[0].date()} to {df_clean.index[-1].date()}')
print(f'N days     : {N}')
print(f'Stations   : {STATIONS}')


Date range : 2010-01-01 to 2025-11-13
N days     : 5796
Stations   : ['AATB', 'JUNG', 'KERG', 'NEWK', 'OULU', 'SOPO', 'THUL', 'JUNG1', 'MXCO', 'APTY', 'ATHN', 'FSMT', 'INVK', 'NAIN', 'PSNM', 'BKSN']


In [3]:
def preprocess_cr(df_in, clip_sigma=3.0, trend_window=365):
    df = df_in.copy()
    roll_med = df.rolling(30, center=True, min_periods=10).median()
    roll_std = df.rolling(30, center=True, min_periods=10).std()
    upper    = roll_med + clip_sigma * roll_std
    lower    = roll_med - clip_sigma * roll_std
    df_clipped = df.copy()
    for col in df.columns:
        mask = (df[col] > upper[col]) | (df[col] < lower[col])
        df_clipped.loc[mask, col] = np.nan
        df_clipped[col] = df_clipped[col].interpolate(method='linear')
    trend = df_clipped.rolling(trend_window, center=True, min_periods=180).mean()
    trend = trend.ffill().bfill()
    df_detrend = df_clipped - trend
    station_means = df_clipped.mean()
    df_proc = df_detrend / station_means * 100
    return df_proc, trend

df_norm, df_trend = preprocess_cr(df_clean)
print('Preprocessing complete.')
print('Post-processing stats (should be ~zero mean):')
for st in ['ATHN', 'JUNG', 'OULU', 'APTY']:
    s = df_norm[st]
    print(f'  {st}: mean={s.mean():+.3f}%  std={s.std():.3f}%')


Preprocessing complete.
Post-processing stats (should be ~zero mean):
  ATHN: mean=+0.002%  std=0.921%
  JUNG: mean=+0.005%  std=1.408%
  OULU: mean=+0.005%  std=1.360%
  APTY: mean=+0.006%  std=1.483%


## 2. MF-DFA parameters and functions

In [4]:
WIN_SIZES_STATIC = fu.powRangeByCount(3, 10, 15, base=2)  # 8 to 1024 days
WIN_SIZES_ROLL   = fu.powRangeByCount(3,  7, 10, base=2)  # 8 to 128 days
QS      = np.arange(-5, 5.1, 0.25)
REV_SEG = True
POL_ORD = 2   # DFA2 -- matches Varotsos (2025)

WINDOW_DAYS = 730  # 2-year sliding window
STEP_DAYS   = 1

print(f'Static win sizes : {WIN_SIZES_STATIC}')
print(f'Rolling win sizes: {WIN_SIZES_ROLL}')
print(f'q range: [{QS[0]:.1f}, {QS[-1]:.1f}] step=0.25')
print(f'Rolling window: {WINDOW_DAYS} days, step: {STEP_DAYS} day')
print(f'Expected rolling windows: ~{N - WINDOW_DAYS}')


Static win sizes : [   8   16   32   64  128  256  512 1024]
Rolling win sizes: [  8  16  32  64 128]
q range: [-5.0, 5.0] step=0.25
Rolling window: 730 days, step: 1 day
Expected rolling windows: ~5066


In [5]:
def run_mfdfa(series, win_sizes=WIN_SIZES_STATIC, qs=QS,
              rev_seg=REV_SEG, pol_ord=POL_ORD):
    """Static MF-DFA. Returns dict with h2, delta_alpha and full spectrum."""
    x = np.array(series, dtype=float)
    x = fu.toAggregated(x)
    obj = fathon.MFDFA(x)
    n, F = obj.computeFlucVec(win_sizes, qs, revSeg=rev_seg, polOrd=pol_ord)
    list_H, list_H_intercept = obj.fitFlucVec()
    tau   = obj.computeMassExponents()
    alpha, f_alpha = obj.computeMultifractalSpectrum()
    h2 = list_H[np.argmin(np.abs(qs - 2))]
    valid_mask = np.isfinite(alpha) & np.isfinite(f_alpha) & (f_alpha >= -0.5)
    delta_a = (alpha[valid_mask].max() - alpha[valid_mask].min()
               if valid_mask.sum() >= 3 else np.nan)
    return {'n':n,'F':F,'qs':qs,'list_H':list_H,
            'list_H_intercept':list_H_intercept,
            'tau':tau,'alpha':alpha,'f_alpha':f_alpha,
            'h2':h2,'delta_alpha':delta_a}


def rolling_mfdfa(series, window=WINDOW_DAYS, step=STEP_DAYS,
                  win_sizes=WIN_SIZES_ROLL, qs=QS,
                  rev_seg=REV_SEG, pol_ord=POL_ORD):
    """Sliding-window MF-DFA. Returns DataFrame with date, h2, delta_alpha."""
    x     = np.array(series, dtype=float)
    dates = series.index if hasattr(series, 'index') else np.arange(len(x))
    results = []
    starts  = range(0, len(x) - window + 1, step)
    for i, start in enumerate(starts):
        seg = x[start:start + window]
        center_date = dates[start + window // 2]
        if np.isnan(seg).sum() > window * 0.05:
            results.append({'date':center_date,'h2':np.nan,'delta_alpha':np.nan})
            continue
        seg = pd.Series(seg).interpolate().values
        try:
            seg_agg = fu.toAggregated(seg)
            obj = fathon.MFDFA(seg_agg)
            n_, F_ = obj.computeFlucVec(win_sizes, qs, revSeg=rev_seg, polOrd=pol_ord)
            list_H_, _ = obj.fitFlucVec()
            alpha_, f_alpha_ = obj.computeMultifractalSpectrum()
            h2_ = list_H_[np.argmin(np.abs(qs - 2))]
            valid = np.isfinite(alpha_) & np.isfinite(f_alpha_) & (f_alpha_ >= -0.5)
            delta_a_ = (alpha_[valid].max() - alpha_[valid].min()
                        if valid.sum() >= 3 else np.nan)
            results.append({'date':center_date,'h2':h2_,'delta_alpha':delta_a_})
        except Exception:
            results.append({'date':center_date,'h2':np.nan,'delta_alpha':np.nan})
        if (i+1) % 500 == 0:
            print(f'  {i+1}/{len(starts)} windows...', end='\r')
    return pd.DataFrame(results).set_index('date')


print('MF-DFA functions defined.')


MF-DFA functions defined.


## 3. NTA functions (kappa1 and S)

NTA runs on `df_clean` (raw interpolated counts), not detrended series.
This matches Varotsos (2025) who uses the unmodified CR intensity sequence.


In [6]:
def compute_nta_kappa1(series_window):
    """NTA variance kappa1. Following Varotsos et al. (2002, 2012)."""
    Q = np.array(series_window, dtype=float)
    Q = Q - Q.min() + 1e-10  # ensure all positive
    N = len(Q)
    chi = np.arange(1, N + 1) / N
    pk  = Q / Q.sum()
    return np.sum(chi**2 * pk) - np.sum(chi * pk)**2


def compute_nta_entropy(series_window):
    """NTA entropy S = <chi ln chi> - <chi> ln <chi>."""
    Q = np.array(series_window, dtype=float)
    Q = Q - Q.min() + 1e-10
    N = len(Q)
    chi = np.arange(1, N + 1) / N
    pk  = Q / Q.sum()
    chi_mean = np.sum(chi * pk)
    return np.sum(chi * np.log(chi) * pk) - chi_mean * np.log(chi_mean)


def rolling_nta(series, window=WINDOW_DAYS, step=STEP_DAYS):
    """Sliding-window NTA. Returns DataFrame with date, kappa1, S_nta."""
    x     = np.array(series, dtype=float)
    dates = series.index
    results = []
    starts = range(0, len(x) - window + 1, step)
    for i, start in enumerate(starts):
        seg = pd.Series(x[start:start + window]).interpolate().values
        center_date = dates[start + window // 2]
        try:
            k1 = compute_nta_kappa1(seg)
            S  = compute_nta_entropy(seg)
        except Exception:
            k1, S = np.nan, np.nan
        results.append({'date':center_date,'kappa1':k1,'S_nta':S})
        if (i+1) % 500 == 0:
            print(f'  {i+1}/{len(starts)} windows...', end='\r')
    return pd.DataFrame(results).set_index('date')


# Sanity check
Su = np.log(2)/2 - 0.25
test_u = np.ones(730)
print(f'Uniform distribution: kappa1=1/12={1/12:.4f}, S_u={Su:.4f}')
print(f'Test (uniform): kappa1={compute_nta_kappa1(test_u):.4f}, '
      f'S={compute_nta_entropy(test_u):.4f}')
print('NTA functions defined.')


Uniform distribution: kappa1=1/12=0.0833, S_u=0.0966
Test (uniform): kappa1=0.0833, S=0.0964
NTA functions defined.


## 4. Static MF-DFA -- all stations
Saves: `Tables_datas/static_mfdfa.csv`


In [7]:
# q-stability check
def check_q_stability(series, qs=QS, threshold=3.0):
    x = fu.toAggregated(np.array(series, dtype=float))
    obj = fathon.MFDFA(x)
    n_, F_ = obj.computeFlucVec(WIN_SIZES_STATIC, qs, revSeg=REV_SEG, polOrd=POL_ORD)
    list_H_, _ = obj.fitFlucVec()
    max_h_neg = float(np.nanmax(np.abs(list_H_[qs < 0])))
    return max_h_neg < threshold, max_h_neg


print('Running static MF-DFA + q-stability for all stations...')
static_results = {}
STABLE_STATIONS = []
static_rows = []

for st in STATIONS:
    stable, mhn = check_q_stability(df_norm[st])
    res = run_mfdfa(df_norm[st])
    static_results[st] = res
    delta_a = res['delta_alpha'] if stable else np.nan
    static_rows.append({
        'station'   : st,
        'h2'        : round(res['h2'], 4),
        'delta_alpha': round(delta_a, 4) if not np.isnan(delta_a) else np.nan,
        'q_stable'  : stable,
        'max_h_neg' : round(mhn, 3),
        'source'    : 'Varotsos (2025)' if st in ['ATHN','JUNG'] else 'This work'
    })
    if stable:
        STABLE_STATIONS.append(st)
    flag = 'OK' if stable else 'ARTEFACT'
    print(f'  {st:8s}: h2={res["h2"]:.4f}  da={delta_a:.4f}  [{flag}]')


df_static = pd.DataFrame(static_rows).set_index('station')
df_static.to_csv('Tables_datas/static_mfdfa.csv')

net_mean_h2 = df_static['h2'].mean()
net_std_h2  = df_static['h2'].std()
net_mean_da = df_static['delta_alpha'].dropna().mean()
net_std_da  = df_static['delta_alpha'].dropna().std()
print(f'\nNetwork: <h(2)>={net_mean_h2:.3f}+/-{net_std_h2:.3f}  '
      f'<Delta-alpha>={net_mean_da:.3f}+/-{net_std_da:.3f}')
print(f'Stable stations ({len(STABLE_STATIONS)}): {STABLE_STATIONS}')
print('Saved: Tables_datas/static_mfdfa.csv')


Running static MF-DFA + q-stability for all stations...
  AATB    : h2=0.9968  da=1.1447  [OK]
  JUNG    : h2=0.9880  da=0.4516  [OK]
  KERG    : h2=0.9793  da=1.5407  [OK]
  NEWK    : h2=1.0106  da=1.6801  [OK]
  OULU    : h2=0.9894  da=0.5277  [OK]
  SOPO    : h2=1.0100  da=2.0756  [OK]
  THUL    : h2=1.0053  da=1.6221  [OK]
  JUNG1   : h2=1.2016  da=0.5464  [OK]
  MXCO    : h2=0.9804  da=1.7218  [OK]
  APTY    : h2=1.0050  da=0.6157  [OK]
  ATHN    : h2=1.0275  da=1.2960  [OK]
  FSMT    : h2=1.0490  da=1.7129  [OK]
  INVK    : h2=0.9841  da=1.5657  [OK]
  NAIN    : h2=1.0196  da=1.7413  [OK]
  PSNM    : h2=0.9666  da=0.9777  [OK]
  BKSN    : h2=0.7533  da=2.3536  [OK]

Network: <h(2)>=0.998+/-0.085  <Delta-alpha>=1.348+/-0.582
Stable stations (16): ['AATB', 'JUNG', 'KERG', 'NEWK', 'OULU', 'SOPO', 'THUL', 'JUNG1', 'MXCO', 'APTY', 'ATHN', 'FSMT', 'INVK', 'NAIN', 'PSNM', 'BKSN']
Saved: Tables_datas/static_mfdfa.csv


## 5. Rolling MF-DFA -- all stations
Saves: `Tables_datas/rolling_mfdfa_{ST}.csv` for each station.
Caches automatically -- skips if file already exists.


In [8]:
roll_all = {}

for st in STATIONS:
    out_file = f'Tables_datas/rolling_mfdfa_{st}.csv'
    if os.path.exists(out_file):
        roll_all[st] = pd.read_csv(out_file, index_col='date', parse_dates=True)
        print(f'{st}: loaded from cache ({len(roll_all[st])} windows)')
    else:
        print(f'Computing rolling MF-DFA for {st}...')
        roll_all[st] = rolling_mfdfa(df_norm[st])
        roll_all[st].to_csv(out_file)
        print(f'  {st}: saved -> {out_file}')

roll_ATHN = roll_all['ATHN']
roll_JUNG = roll_all['JUNG']
print(f'\nAll rolling MF-DFA done. ATHN: {len(roll_ATHN)} windows.')


AATB: loaded from cache (5067 windows)
JUNG: loaded from cache (5067 windows)
KERG: loaded from cache (5067 windows)
NEWK: loaded from cache (5067 windows)
OULU: loaded from cache (5067 windows)
SOPO: loaded from cache (5067 windows)
THUL: loaded from cache (5067 windows)
JUNG1: loaded from cache (5067 windows)
MXCO: loaded from cache (5067 windows)
APTY: loaded from cache (5067 windows)
ATHN: loaded from cache (5067 windows)
FSMT: loaded from cache (5067 windows)
INVK: loaded from cache (5067 windows)
NAIN: loaded from cache (5067 windows)
PSNM: loaded from cache (5067 windows)
BKSN: loaded from cache (5067 windows)

All rolling MF-DFA done. ATHN: 5067 windows.


## 6. Rolling NTA -- ATHN and JUNG
Saves: `Tables_datas/rolling_nta_ATHN.csv`, `Tables_datas/rolling_nta_JUNG.csv`

NTA runs on `df_clean` (raw interpolated counts) -- NOT detrended.
This is consistent with Varotsos (2025).


In [9]:
for st in ['ATHN', 'JUNG']:
    out_file = f'Tables_datas/rolling_nta_{st}.csv'
    if os.path.exists(out_file):
        print(f'NTA {st}: loaded from cache')
    else:
        print(f'Computing rolling NTA for {st}...')
        nta = rolling_nta(df_clean[st])
        nta.to_csv(out_file)
        print(f'  Saved: {out_file} ({len(nta)} windows)')

nta_ATHN = pd.read_csv('Tables_datas/rolling_nta_ATHN.csv',
                       index_col='date', parse_dates=True)
nta_JUNG = pd.read_csv('Tables_datas/rolling_nta_JUNG.csv',
                       index_col='date', parse_dates=True)
print(f'NTA ATHN: {len(nta_ATHN)} windows | NTA JUNG: {len(nta_JUNG)} windows')
print(f'ATHN kappa1: mean={nta_ATHN["kappa1"].mean():.4f}, '
      f'min={nta_ATHN["kappa1"].min():.4f}, max={nta_ATHN["kappa1"].max():.4f}')


NTA ATHN: loaded from cache
NTA JUNG: loaded from cache
NTA ATHN: 5067 windows | NTA JUNG: 5067 windows
ATHN kappa1: mean=0.0814, min=0.0702, max=0.0952


## 7. Combined tables (MF-DFA + NTA merged)
Saves: `Tables_datas/combined_ATHN.csv`, `Tables_datas/combined_JUNG.csv`


In [10]:
for st, roll_df, nta_df in [
    ('ATHN', roll_ATHN, nta_ATHN),
    ('JUNG', roll_JUNG, nta_JUNG)
]:
    combined = pd.merge(
        roll_df[['h2', 'delta_alpha']],
        nta_df[['kappa1', 'S_nta']],
        left_index=True, right_index=True, how='inner'
    ).dropna()
    combined.to_csv(f'Tables_datas/combined_{st}.csv')
    print(f'combined_{st}: {len(combined)} windows -> Tables_datas/combined_{st}.csv')
    print(f'  columns: {list(combined.columns)}')
    print(combined.describe().round(4))
    print()


combined_ATHN: 5067 windows -> Tables_datas/combined_ATHN.csv
  columns: ['h2', 'delta_alpha', 'kappa1', 'S_nta']
              h2  delta_alpha     kappa1      S_nta
count  5067.0000    5067.0000  5067.0000  5067.0000
mean      1.0666       1.4148     0.0814     0.0969
std       0.0819       0.7816     0.0053     0.0078
min       0.8723       0.0379     0.0702     0.0770
25%       1.0041       0.6720     0.0771     0.0927
50%       1.0833       1.5056     0.0816     0.0959
75%       1.1239       2.0898     0.0843     0.1017
max       1.2271       3.0778     0.0952     0.1210

combined_JUNG: 5067 windows -> Tables_datas/combined_JUNG.csv
  columns: ['h2', 'delta_alpha', 'kappa1', 'S_nta']
              h2  delta_alpha     kappa1      S_nta
count  5067.0000    5067.0000  5067.0000  5067.0000
mean      1.0794       0.6528     0.0804     0.0944
std       0.0850       0.3845     0.0057     0.0091
min       0.8926       0.1579     0.0659     0.0645
25%       1.0018       0.4282     0.0766   

## 8. FD parameter table
Saves: `Tables_datas/fd_parameters.csv`

Extracts h(2), Delta-alpha, kappa1, S at each FD onset (+/-3 days tolerance).


In [11]:
combined_ATHN = pd.read_csv('Tables_datas/combined_ATHN.csv',
                            index_col='date', parse_dates=True)
combined_JUNG = pd.read_csv('Tables_datas/combined_JUNG.csv',
                            index_col='date', parse_dates=True)

def closest_val(df_roll, col, fd_date, tol_days=3):
    t0 = fd_date - pd.Timedelta(days=tol_days)
    t1 = fd_date + pd.Timedelta(days=tol_days)
    w = df_roll.loc[t0:t1, col].dropna()
    if len(w) == 0: return np.nan
    return float(w.iloc[np.abs((w.index - fd_date).total_seconds()).argmin()])


records = []
for fd_date, row in df_fd.iterrows():
    records.append({
        'date'         : fd_date,
        'magnitude'    : row['magnitude'],
        'ATHN_h2'      : closest_val(combined_ATHN, 'h2',          fd_date),
        'ATHN_da'      : closest_val(combined_ATHN, 'delta_alpha', fd_date),
        'ATHN_kappa1'  : closest_val(combined_ATHN, 'kappa1',      fd_date),
        'ATHN_S'       : closest_val(combined_ATHN, 'S_nta',       fd_date),
        'JUNG_h2'      : closest_val(combined_JUNG, 'h2',          fd_date),
        'JUNG_da'      : closest_val(combined_JUNG, 'delta_alpha', fd_date),
        'JUNG_kappa1'  : closest_val(combined_JUNG, 'kappa1',      fd_date),
        'JUNG_S'       : closest_val(combined_JUNG, 'S_nta',       fd_date),
    })

df_fd_params = pd.DataFrame(records).set_index('date')
df_fd_params.to_csv('Tables_datas/fd_parameters.csv')
print('=== FD parameter table ===')
print(df_fd_params.round(4).to_string())
print('\nSaved: Tables_datas/fd_parameters.csv')


=== FD parameter table ===
            magnitude  ATHN_h2  ATHN_da  ATHN_kappa1  ATHN_S  JUNG_h2  JUNG_da  JUNG_kappa1  JUNG_S
date                                                                                               
2011-10-24       6.01   1.1548   0.8926       0.0800  0.0922   1.1625   0.7965       0.0829  0.1015
2012-03-08      10.40   1.1532   1.0630       0.0795  0.0907   1.1603   0.7586       0.0851  0.1001
2012-07-14       6.70   1.1714   1.0680       0.0790  0.0955   1.1584   0.7118       0.0819  0.0985
2015-06-22       8.57   1.0696   2.7424       0.0878  0.1012   1.1704   1.7350       0.0896  0.0983
2017-09-07       6.41   1.0486   0.5215       0.0748  0.0854   0.9889   0.5530       0.0879  0.0945
2021-11-03       9.68   1.0399   1.7521       0.0706  0.0914   1.0437   0.5838       0.0708  0.0923
2023-04-23       6.57   1.0884   1.7329       0.0907  0.1121   1.0843   0.6486       0.0717  0.0988
2024-03-24      11.47   1.1525   2.2470       0.0741  0.0931   1.1550   0

## 10. Surrogate testing for multifractality origin

Addresses Varotsos comment: what drives the observed multifractality --
long-range **correlations** or **broad/non-Gaussian PDF**?

We generate two types of surrogates for each station:

- **Phase Randomization (PR)**: shuffles Fourier phases randomly.
  Destroys long-range correlations but *preserves the PDF exactly*.
  If Δα(original) >> Δα(PR), correlations contribute to multifractality.

- **AAFT (Amplitude Adjusted Fourier Transform)**: rescales the
  phase-randomized surrogate back to the original rank-order values.
  Preserves both the PDF and approximately the power spectrum.
  If Δα(original) >> Δα(AAFT), the excess multifractality is due to
  nonlinear correlations beyond the linear power spectrum.

Saves: `Tables_datas/surrogate_results.csv`
       `Tables_datas/surrogate_detail.csv` (all surrogate Δα values)


In [12]:
def make_pr_surrogate(series):
    """Phase Randomization surrogate.
    Destroys correlations, preserves PDF.
    """
    x = np.array(series, dtype=float)
    n = len(x)
    # FFT
    ft = np.fft.rfft(x)
    # Randomize phases (preserve conjugate symmetry)
    phases = np.random.uniform(0, 2*np.pi, len(ft))
    # Keep DC and Nyquist real
    phases[0] = 0
    if n % 2 == 0:
        phases[-1] = 0
    ft_surr = np.abs(ft) * np.exp(1j * phases)
    surr = np.fft.irfft(ft_surr, n=n)
    return surr


def make_aaft_surrogate(series):
    """Amplitude Adjusted Fourier Transform surrogate.
    Preserves PDF (rank-order) and approximately the power spectrum.
    """
    x = np.array(series, dtype=float)
    n = len(x)
    # Rank-order map from Gaussian to original
    gaussian = np.random.normal(0, 1, n)
    rank_orig = np.argsort(np.argsort(x))      # ranks of original
    rank_gauss = np.argsort(np.argsort(gaussian))  # ranks of gaussian
    # Map: replace Gaussian values at positions sorted like original
    x_gauss = np.empty(n)
    x_gauss[rank_orig] = np.sort(gaussian)[rank_gauss]
    # Phase randomize the Gaussian version
    pr = make_pr_surrogate(x_gauss)
    # Re-rank back to original amplitudes
    rank_pr = np.argsort(np.argsort(pr))
    surr = np.empty(n)
    surr[rank_pr] = np.sort(x)[np.argsort(rank_pr)]
    return surr


def delta_alpha_from_series(series, win_sizes=WIN_SIZES_STATIC):
    """Compute Delta-alpha only (fast path for surrogate loop)."""
    x = fu.toAggregated(np.array(series, dtype=float))
    obj = fathon.MFDFA(x)
    n_, F_ = obj.computeFlucVec(win_sizes, QS, revSeg=REV_SEG, polOrd=POL_ORD)
    _ = obj.fitFlucVec()
    alpha_, f_ = obj.computeMultifractalSpectrum()
    valid = np.isfinite(alpha_) & np.isfinite(f_) & (f_ >= -0.5)
    return (alpha_[valid].max() - alpha_[valid].min()
            if valid.sum() >= 3 else np.nan)


print('Surrogate functions defined.')
print('PR: destroys correlations, preserves PDF')
print('AAFT: preserves PDF + power spectrum approximately')


Surrogate functions defined.
PR: destroys correlations, preserves PDF
AAFT: preserves PDF + power spectrum approximately


In [13]:
out_file   = 'Tables_datas/surrogate_results.csv'
out_detail = 'Tables_datas/surrogate_detail.csv'

if os.path.exists(out_file):
    print('Loaded from cache:', out_file)
else:
    np.random.seed(42)
    N_SURR = 1000  # number of surrogates per type per station

    summary_rows = []
    detail_rows  = []

    for st in STATIONS:
        series = df_norm[st].dropna().values
        print(f'\nProcessing {st} ({len(series)} days)...')

        # Original Delta-alpha (already computed in static_results)
        da_orig = float(df_static.loc[st, 'delta_alpha'])

        # PR surrogates
        da_pr = []
        for i in range(N_SURR):
            try:
                surr = make_pr_surrogate(series)
                da_pr.append(delta_alpha_from_series(surr))
            except Exception:
                da_pr.append(np.nan)
            if (i+1) % 200 == 0:
                print(f'  PR {i+1}/{N_SURR}', end='\r')
        da_pr = np.array(da_pr)

        # AAFT surrogates
        da_aaft = []
        for i in range(N_SURR):
            try:
                surr = make_aaft_surrogate(series)
                da_aaft.append(delta_alpha_from_series(surr))
            except Exception:
                da_aaft.append(np.nan)
            if (i+1) % 200 == 0:
                print(f'  AAFT {i+1}/{N_SURR}', end='\r')
        da_aaft = np.array(da_aaft)

        # p-values: fraction of surrogates with Da >= Da_original
        p_pr   = (da_pr[~np.isnan(da_pr)]   >= da_orig).mean()
        p_aaft = (da_aaft[~np.isnan(da_aaft)] >= da_orig).mean()

        # z-scores
        z_pr   = (da_orig - np.nanmean(da_pr))   / np.nanstd(da_pr)
        z_aaft = (da_orig - np.nanmean(da_aaft)) / np.nanstd(da_aaft)

        summary_rows.append({
            'station'      : st,
            'da_original'  : round(da_orig, 4),
            'da_pr_mean'   : round(np.nanmean(da_pr),   4),
            'da_pr_std'    : round(np.nanstd(da_pr),    4),
            'da_aaft_mean' : round(np.nanmean(da_aaft), 4),
            'da_aaft_std'  : round(np.nanstd(da_aaft),  4),
            'z_pr'         : round(z_pr,   3),
            'z_aaft'       : round(z_aaft, 3),
            'p_pr'         : round(p_pr,   4),
            'p_aaft'       : round(p_aaft, 4),
        })

        # Store all surrogate values for NB2 plotting
        for val in da_pr:
            detail_rows.append({'station': st, 'type': 'PR',   'da': val})
        for val in da_aaft:
            detail_rows.append({'station': st, 'type': 'AAFT', 'da': val})

        print(f'  {st}: da_orig={da_orig:.4f} | '
              f'PR: mean={np.nanmean(da_pr):.4f} z={z_pr:.2f} p={p_pr:.4f} | '
              f'AAFT: mean={np.nanmean(da_aaft):.4f} z={z_aaft:.2f} p={p_aaft:.4f}')

    df_surr    = pd.DataFrame(summary_rows).set_index('station')
    df_detail  = pd.DataFrame(detail_rows)
    df_surr.to_csv(out_file)
    df_detail.to_csv(out_detail, index=False)
    print(f'\nSaved: {out_file}')
    print(f'Saved: {out_detail}')

df_surr   = pd.read_csv(out_file,   index_col='station')
df_detail = pd.read_csv(out_detail)
print('\n=== Surrogate test summary ===')
print(df_surr[['da_original','da_pr_mean','da_aaft_mean',
               'z_pr','z_aaft','p_pr','p_aaft']].round(4).to_string())



Processing AATB (5796 days)...
  AATB: da_orig=1.1447 | PR: mean=0.3272 z=13.56 p=0.0000 | AAFT: mean=0.2977 z=16.78 p=0.0000

Processing JUNG (5796 days)...
  JUNG: da_orig=0.4516 | PR: mean=0.3088 z=2.36 p=0.0200 | AAFT: mean=0.3125 z=2.65 p=0.0170

Processing KERG (5796 days)...
  KERG: da_orig=1.5407 | PR: mean=0.3057 z=20.90 p=0.0000 | AAFT: mean=0.3142 z=22.50 p=0.0000

Processing NEWK (5796 days)...
  NEWK: da_orig=1.6801 | PR: mean=0.3092 z=21.22 p=0.0000 | AAFT: mean=0.2929 z=27.25 p=0.0000

Processing OULU (5796 days)...
  OULU: da_orig=0.5277 | PR: mean=0.3162 z=3.57 p=0.0020 | AAFT: mean=0.3127 z=4.00 p=0.0010

Processing SOPO (5796 days)...
  SOPO: da_orig=2.0756 | PR: mean=0.3283 z=27.99 p=0.0000 | AAFT: mean=0.3080 z=32.49 p=0.0000

Processing THUL (5796 days)...
  THUL: da_orig=1.6221 | PR: mean=0.3300 z=20.62 p=0.0000 | AAFT: mean=0.2944 z=24.79 p=0.0000

Processing JUNG1 (5796 days)...
  JUNG1: da_orig=0.5464 | PR: mean=0.3390 z=3.52 p=0.0020 | AAFT: mean=0.2300 z=6.

## 9. File summary

In [14]:
print('=== Tables_datas/ contents ===')
for f in sorted(os.listdir('Tables_datas')):
    kb = os.path.getsize(f'Tables_datas/{f}') / 1024
    print(f'  {f:<50s} {kb:7.1f} KB')
print('\nAll done. Run NB2_AnalysisFigures to reproduce all figures.')


=== Tables_datas/ contents ===
  .ipynb_checkpoints                                     0.3 KB
  combined_ATHN.csv                                    426.4 KB
  combined_JUNG.csv                                    427.2 KB
  fd_parameters.csv                                      1.6 KB
  robustness_detrending_ATHN.csv                       612.1 KB
  robustness_detrending_JUNG.csv                       616.1 KB
  robustness_interpolation.csv                           0.2 KB
  robustness_roll_ATHN_trend180.csv                    240.4 KB
  robustness_roll_ATHN_trend365.csv                    240.3 KB
  robustness_roll_ATHN_trend730.csv                    240.2 KB
  robustness_roll_JUNG_trend180.csv                    241.7 KB
  robustness_roll_JUNG_trend365.csv                    241.7 KB
  robustness_roll_JUNG_trend730.csv                    241.6 KB
  rolling_mfdfa_AATB.csv                               240.5 KB
  rolling_mfdfa_APTY.csv                               241.5 KB
  rolling